In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpdaf.obj import Cube,Image,WaveCoord,Spectrum,deg2sexa,sexa2deg

from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
from astropy.io import fits
from astropy.table import Table
import math

import numpy.ma as ma
%matplotlib inline
from astropy.table import Table, join
from mpdaf.sdetect.source import Source
import pandas as pd
#Binning functions
import spectrum_functions as sf
import bin_plots as bp
import binning_functions as bf
import Main_binning_loop as mb


[INFO] pyplatefit detected. Running with full spectral fitting mode.


### The code gives: 
* A text file: column 1 contains bin number; column 2 contains spaxel coordinates in that bin.
* A plot of binmap


### A note on n_annuli:
* n_annuli : Number of concentric radial rings used to initialize the grid, effectively controlling the initial radial resolution. The code sets up a polar mesh where the number of azimuthal sectors increases by 6 for each outward ring (6, 12, 18, etc.). 
- Note on Trade-off: A lower value significantly speeds up execution time but results in a smaller number of thicker bins with more spaxels each, compromising the final radial resolution of your metallicity measurements. A higher value provides a finer starting mesh for detailed gradients.

In [2]:
cube_file="path to the cube"
#"/home/achougule/IA_postdoc_2024_new/DATA_for_Chougule_et_al_2025/HDFS/Cubes/MHDFS-0003/MHDFS-0003-cube.fits"
cube = Cube(cube_file, ext=('DATA', 'VAR'))
dec_center,ra_center,pa,inc,rd,input_redshift=[-60.56043692358837,338.2239249890238,-18.8,16,0.6597618829636348,0.563749]

n_annuli = 6
sn_threshold = 5

#Location for text file
binmap_location="/home/achougule/IA_postdoc_2024_new/DATA_for_Chougule_et_al_2025/binning_code_github/"
#Location for figures
savefig_location="/home/achougule/IA_postdoc_2024_new/DATA_for_Chougule_et_al_2025/binning_code_github/"
binmap_savefig_location="/home/achougule/IA_postdoc_2024_new/DATA_for_Chougule_et_al_2025/binning_code_github/"


#If pyplatefit is installed
binmap_after_acc=mb.bin_spaxels(cube,dec_center,ra_center, input_redshift,sn_threshold, rd, pa,inc,n_annuli, binmap_savefig_location)

#If pyplatefit is not installed
#observed_emission_lines=['OII3727b', 'HGAMMA', 'HBETA', 'OIII5007']
#observed_wavelengths=[5831.577351076927, 6789.997283646275, 7604.799174797293, 7833.207195372561]
#observed_FWHM = [2.943685832664074, 4.893285634910402, 5.335917188248487, 3.1565230133550717]

#binmap_after_acc=mb.bin_spaxels(cube,dec_center,ra_center, input_redshift,sn_threshold, rd, pa,inc,n_annuli, binmap_savefig_location,
                #observed_emission_lines, observed_wavelengths, observed_FWHM)

[INFO] Running bin_spaxels in Mode A (pyplatefit spectral fitting).
[DEBUG] Performing a first quick fit to refine the input redshift


[DEBUG] Preparing data for fit
[DEBUG] Getting lines from default line table...
[DEBUG] 19.0 % of the spectrum is used for fitting.
[DEBUG] Initialize fit
[DEBUG] Init fit all lines together expect Lya
[DEBUG] Found 18 lines to fit
[DEBUG] added 18 gaussian to the fit
[DEBUG] Fitting of Line Family: all
[DEBUG] Lmfit fitting: {'method': 'least_square', 'xtol': 0.001}
[DEBUG] `xtol` termination condition is satisfied. after 231 iterations, reached minimum = 27086.810 and redChi2 = 40.248
[DEBUG] Saving all results to tablines and ztab
[DEBUG] Computed velocity offset 144.8 km/s
[DEBUG] Fit continuum
[DEBUG] Fit emission lines
[DEBUG] Preparing data for fit
[DEBUG] Getting lines from default line table...
[DEBUG] 19.1 % of the spectrum is used for fitting.
[DEBUG] Initialize fit
[DEBUG] Found 2 non resonnant line families to fit
[DEBUG] Init Fit of family forbidden
[DEBUG] Found 10 lines to fit
[DEBUG] added 10 gaussian to the fit
[DEBUG] Init Fit of family balmer
[DEBUG] Found 8 lines t

In [3]:
# Open the file to write the multi-spaxel bin data
with open(binmap_location+"bin_spaxel_map.txt", "w") as f:
    f.write("# Bin_ID   Spaxel_Coordinates_[x,y]\n")
    
    for row in binmap_after_acc[:, 0:2]:
        bin_num = row[0]
        
        # Filter out the -1 (unbinned/rejected) spaxels
        if bin_num == -1:
            continue
            
        # row[1] is your list of coordinates: [[x1, y1], [x2, y2], ...]
        # We join them into a string format: [x1,y1],[x2,y2]
        spaxel_list = row[1]
        coord_string = ",".join([f"[{c[0]},{c[1]}]" for c in spaxel_list])
        
        # Write the bin number and all its associated spaxels on one line
        f.write(f"{bin_num:<8} {coord_string}\n")

print("Finished writing bin_spaxel_map.txt!")

Finished writing bin_spaxel_map.txt!
